<a href="https://colab.research.google.com/github/umaimakhalid17/ML/blob/main/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/umaimakhalid17/ML/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
!git clone https://github.com/umaimakhalid17/ML.git
%cd ML/work/notebooks

Cloning into 'ML'...
remote: Enumerating objects: 119, done.
remote: Counting objects: 100% (119/119), done.
remote: Compressing objects: 100% (72/72), done.
remote: Total 119 (delta 32), reused 104 (delta 31), pack-reused 0 (from 0)
Receiving objects: 100% (119/119), 1.84 MiB | 7.63 MiB/s, done.
Resolving deltas: 100% (32/32), done.
/content/ML/work/notebooks


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task type: Clustering.**

My lane is Structured Content Archetype Clustering (Lane 3) — the question is "What performance archetypes exist across the content inventory?" That's not classification (no known categories exist ahead of time), not ranking (I'm not ordering pages by priority — that's Lane 2 and Lane 4's job), and not regression/forecasting (I'm not predicting a number). Nobody has told me in advance what groups exist or how many. I want the structure — pages that behave alike on traffic, position, freshness, and depth — to emerge from the data itself. That's the definition of clustering: unsupervised grouping, not prediction against a known answer.

In [ ]:
task_type = "clustering"
lane = "Lane 3 -- Structured Content Archetype Clustering"

print(f"Lane: {lane}")
print(f"ML task type: {task_type}")


Lane: Lane 3 -- Structured Content Archetype Clustering
ML task type: clustering


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

There is **no supervised target** in this lane — nothing in the data says "this page's true archetype is X." No archetype ground truth exists to check against, unlike Lane 2's `is_declining_label` (`trend_direction == "down"`), which is a real, checkable observed outcome.

**The proxy is `cluster_id`** — an integer assigned to every page, discovered from where it sits in feature space (traffic, position, freshness, engagement), not predicted from a known answer. Below I load the starter data, build a feature matrix, and run a lightweight illustrative clustering pass to sketch what that `cluster_id` column actually looks like. This is a sketch, not the tuned capstone clustering — the point here is to show the shape of the proxy, not defend a final k.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

pd.set_option("display.max_columns", 60)

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# avg_position == 0 means "no position data," not a real position (data dictionary gotcha)
df["has_position_data"] = df["avg_position"] > 0

# heavy-tailed traffic counts -> log1p, same idea the starter pipeline uses
for c in ["impressions_90d", "clicks_90d", "sessions_90d"]:
    df[f"log_{c}"] = np.log1p(df[c])

sketch_cols = ["log_impressions_90d", "log_clicks_90d", "log_sessions_90d",
               "ctr", "engagement_rate", "scroll_rate"]
sketch_df = df.dropna(subset=sketch_cols).copy()

X = StandardScaler().fit_transform(sketch_df[sketch_cols])

# k=4 is a placeholder for this sketch, not a chosen final value
km = KMeans(n_clusters=4, random_state=42, n_init=10)
sketch_df["cluster_id"] = km.fit_predict(X)

print(sketch_df["cluster_id"].value_counts().sort_index())
sketch_df[["content_id", "content_type", "impressions_90d", "ctr", "cluster_id"]].head(8)


cluster_id
0     9669
1    16161
2      154
3     3891
Name: count, dtype: int64


,content_id,content_type,impressions_90d,ctr,cluster_id
0,content_304f48230142,keyword article,3803,0.76,0
1,content_a1fb4e703a9e,keyword article,15320,0.05,0
2,content_9aa793d4d895,keyword article,12581,0.09,0
3,content_331d6c4de07b,keyword article,11751,0.49,0
4,content_d99b7a2d90ca,keyword article,19140,0.13,0
5,content_d4084a4bc775,keyword article,3970,0.03,1
6,content_9a34b442b552,keyword article,20,0.00,1
7,content_a63219c6e95a,keyword article,1724,0.06,1


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Per `docs/ml-core-foundation-framework.md` (section 8), clustering isn't judged by accuracy — it's judged by **silhouette, stability, and human usefulness**. Three checks, in order of how much I actually trust them:

1. **Silhouette score** — how tight and separated the clusters are geometrically. Computed below for the sketch above; this is the closest thing to a defensible single number, but it only measures shape, not usefulness.
2. **Stability** — do the same archetypes reappear with a different seed or a different k? Not computed in this sketch, but it's the next check before I'd trust any cluster.
3. **Human usefulness** — can I look at a cluster's profile and name it something a content reviewer would recognize ("stale visible page," "hidden gem") per the lane guide's archetype list? This is the metric that actually decides whether the clustering did its job, and it can't be computed — only judged by inspection.

The sketch clustering scored 0.383, clearing the ~0.25–0.3 floor I set above. That's evidence the geometric shape is real, not evidence the clusters are useful yet — stability and human-naming still need to happen before I'd trust this for the capstone.

In [ ]:
from sklearn.metrics import silhouette_score

score = silhouette_score(X, sketch_df["cluster_id"])
print(f"Silhouette score (sketch clustering, k=4): {score:.3f}")


Silhouette score (sketch clustering, k=4): 0.383


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one content item (page).** That's already the grain of `content_refresh_anonymized.csv` — `content_id` is unique per row — so I don't need to reshape anything for this lane, just select and clean the columns that describe what a page currently looks like. Below I show the unit of analysis as a real dataframe, reusing `sketch_df` from section 2 rather than reloading the data twice.

In [ ]:
print("One row =", "one content item (page).")
print("content_id uniqueness check:", sketch_df["content_id"].is_unique)
print(sketch_df.shape)

unit_of_analysis_cols = [
    "content_id", "client_id", "content_type", "main_intent",
    "word_count", "content_age_days", "days_since_last_update",
    "impressions_90d", "clicks_90d", "sessions_90d",
    "ctr", "avg_position", "engagement_rate", "scroll_rate",
    "has_position_data", "cluster_id",
]

sketch_df[unit_of_analysis_cols].head(5)


One row = one content item (page).
content_id uniqueness check: True
(29875, 49)


,content_id,client_id,content_type,main_intent,word_count,content_age_days,days_since_last_update,impressions_90d,clicks_90d,sessions_90d,ctr,avg_position,engagement_rate,scroll_rate,has_position_data,cluster_id
0,content_304f48230142,client_f369cb89fc,keyword article,transactional,3221.0,187,20,3803,29,17,0.76,10.6,5.88,4.55,True,0
1,content_a1fb4e703a9e,client_4e07408562,keyword article,informational,2481.0,445,25,15320,7,9,0.05,20.3,0.00,10.00,True,0
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,informational,3515.0,141,20,12581,11,11,0.09,36.5,0.00,28.57,True,0
3,content_331d6c4de07b,client_19581e27de,keyword article,commercial,NaN,463,22,11751,58,78,0.49,6.2,1.28,3.45,True,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,informational,2803.0,263,14,19140,24,145,0.13,44.0,0.00,24.29,True,0


In [ ]:
tier_cols = ["age_tier", "impression_tier", "position_tier", "freshness_tier"]

possible_combinations = 1
for c in tier_cols:
    possible_combinations *= df[c].nunique()

observed_combinations = df.groupby(tier_cols, dropna=False).ngroups

print(f"Theoretically possible tier combinations: {possible_combinations}")
print(f"Combinations actually observed in this data: {observed_combinations}")


Theoretically possible tier combinations: 320
Combinations actually observed in this data: 175


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

The starter data already ships several fixed, single-dimension rules: `age_tier`, `impression_tier`, `position_tier`, `freshness_tier`. Each buckets a page by exactly one number. A page can be `impression_tier = excellent` AND `position_tier = deep` AND `freshness_tier = never` all at once — that combination is the real story, not any tier alone.

175 of 320 theoretically possible tier combinations actually occur — about 55%, a real reduction but not the combinatorial explosion the raw numbers might suggest alone. The distribution is what makes the case: the top 10 combinations alone account for 35.9% of all pages, while 15 of the 175 observed combinations appear only once. So even within the "realistic" 175, most of the volume concentrates in a handful of patterns — each a specific, multi-dimensional intersection (e.g. 91-180 / low / page_1 / 0-30, 1,747 pages) that a rule-writer would have to notice and hand-code separately — while the long tail is made of one-off cases nobody would think to write a rule for in advance. That's the real argument against a fixed rule: not that there are too many combinations to count, but that the ones that matter are specific multi-dimensional intersections, and the ones that don't matter are too sparse to justify hand-coding.

In [ ]:
combo_sizes = df.groupby(tier_cols, dropna=False).size().sort_values(ascending=False)

print(combo_sizes.describe())
print("\nTop 10 combinations by page count:")
print(combo_sizes.head(10))
print(f"\nShare of pages in top 10 combos: {combo_sizes.head(10).sum() / len(df):.1%}")
print(f"Combos with only 1 page (long-tail): {(combo_sizes == 1).sum()} of {len(combo_sizes)}")

count     175.000000
mean      171.428571
std       297.272359
min         1.000000
25%         5.000000
50%        26.000000
75%       192.000000
max      1747.000000
dtype: float64

Top 10 combinations by page count:
age_tier  impression_tier  position_tier  freshness_tier
91-180    low              page_1         0-30              1747
          moderate         page_1         0-30              1272
181-365   low              page_1         0-30              1191
91-180    good             page_1         0-30              1110
          moderate         striking       0-30              1022
181-365   low              top_3          0-30              1007
365+      moderate         page_3_5       0-30               895
181-365   good             page_1         91-180             884
          moderate         page_3_5       91-180             825
365+      moderate         striking       0-30               825
dtype: int64

Share of pages in top 10 combos: 35.9%
Combos with only 1 pa

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.